## nginx client_max_body_size

```bash
cat /etc/nginx/sites-available/tryit

server {
  server_name tryit.parleyvale.com;
  client_max_body_size 10M;

        root /home/tryit/public_html;
        index index.php index.html index.htm;

        location / {
    try_files $uri $uri/ =404;
    autoindex on;
        }
  location /abv/app/ {
    alias /home/tryit/public_html/abv/app/;
    try_files $uri $uri/ =404;
  }

  location /abv/api/ {
      proxy_pass http://127.0.0.1:5000/api/;
      proxy_set_header Host $host;
      proxy_set_header X-Real-IP $remote_addr;
      proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
      proxy_set_header X-Forwarded-Proto $scheme;
  }

  location = /abv/ {
      return 301 /abv/app/;
  }
  location ~ \.php$ {
    include snippets/fastcgi-php.conf;
    fastcgi_pass unix:/var/run/php/php8.1-fpm.sock;
                }
  location ~ /\.ht {
      deny all;
  }

    listen 443 ssl; # managed by Certbot
    ssl_certificate /etc/letsencrypt/live/parleyvale.com/fullchain.pem; # managed by Certbot
    ssl_certificate_key /etc/letsencrypt/live/parleyvale.com/privkey.pem; # managed by Certbot
    include /etc/letsencrypt/options-ssl-nginx.conf; # managed by Certbot
    ssl_dhparam /etc/letsencrypt/ssl-dhparams.pem; # managed by Certbot


}server {
    if ($host = tryit.parleyvale.com) {
        return 301 https://$host$request_uri;
    } # managed by Certbot


  server_name tryit.parleyvale.com;
  client_max_body_size 10M;
    listen 80;
    return 404; # managed by Certbot

```


## flask allowed file types, max_content_length, 

in app.py lin28
```python
DATABASE = '../songs.db'
UPLOAD_FOLDER = '../resources'
ALLOWED_EXTENSIONS = {'txt', 'pdf', 'png', 'jpg', 'jpeg', 'gif', 'mp3', 'wav', 'm4a', 'aac', 'flac', 'ogg', 'html', 'css', 'js'}

app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 10 * 1024 * 1024  # 10MB max file size (10,000K)
```

### killing server

```python
pkill -f "python.*app.py" && cd /home/tryit/public_html/abv/server && nohup python3 app.py > nohup.out 2>&1 &
```

## staring up both servers on windows

```bash
# Complete startup sequence with nohup
echo "🚀 Starting complete dual server setup with nohup..."

# Step 1: Kill any existing Python processes
echo "Stopping all existing Python processes..."
taskkill //F //IM python.exe 2>/dev/null || echo "No Python processes found"
sleep 2

# Step 2: Start static file server on port 8000 (background)
echo "Starting static file server on port 8000..."
cd "C:/Users/mcken/OneDrive/chorus/abv/tech"
python -m http.server 8000 > static.log 2>&1 &
STATIC_PID=$!
echo "Static server PID: $STATIC_PID"
sleep 2

# Step 3: Start Flask API server with nohup (should get port 8001)
echo "Starting Flask API server with nohup..."
cd "C:/Users/mcken/OneDrive/chorus/abv/tech/abv/server"
nohup "C:\Users\mcken\miniconda3\python.exe" app.py > nohup.out 2>&1 &
FLASK_PID=$!
echo "Flask server PID: $FLASK_PID"
sleep 3

# Step 4: Verify both servers are running
echo ""
echo "🔍 Checking server status:"
netstat -an | findstr ":8000" && echo "✅ Port 8000: Static files running"
netstat -an | findstr ":8001" && echo "✅ Port 8001: Flask API running"

echo ""
echo "📜 Check Flask logs: tail -f nohup.out"
echo "🌐 Test URLs:"
echo "   Static App: http://localhost:8000/abv/app/"
echo "   Flask API:  http://127.0.0.1:8001/api/songs"
```